# Telemachus v0.8 — AEGIS demo

[![PyPI](https://img.shields.io/pypi/v/telemachus.svg)](https://pypi.org/project/telemachus/)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19609044.svg)](https://doi.org/10.5281/zenodo.19609044)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/telemachus3/telemachus/blob/main/docs/notebooks/aegis-demo.ipynb)

A 5-minute tour of the Telemachus format using the AEGIS open dataset
(Graz, Austria — 33 trips, 1.06M rows, GNSS + IMU + gyroscope, CC-BY-4.0).

**What we will do**

1. Install `telemachus` from PyPI
2. Download one Parquet file from Zenodo
3. Load with `tele.read()` — get a pandas DataFrame with SI units
4. Inspect sensors available on the device
5. Visualise one trip: trajectory + speed + acceleration + gyroscope
6. Validate against the Telemachus v0.8 schema

No configuration, no proprietary layer — just the open format.

In [ ]:
%pip install -q telemachus matplotlib

In [ ]:
# Download the AEGIS Telemachus parquet (~17 MB) from Zenodo
import pathlib, urllib.request

url = "https://zenodo.org/records/19609044/files/aegis-telemachus.parquet"
path = pathlib.Path("aegis-telemachus.parquet")
if not path.exists():
    urllib.request.urlretrieve(url, path)
print(f"{path.name}: {path.stat().st_size / 1e6:.1f} MB")

In [ ]:
import telemachus as tele

df = tele.read("aegis-telemachus.parquet")
print(f"telemachus {tele.__version__} | rows={len(df):,} | trips={df['trip_id'].nunique()} | profile={tele.sensor_profile(df)}")
df.head(3)

## Sensor introspection

Telemachus exposes helpers to check what the device actually recorded,
without having to peek at column names by hand.

In [ ]:
checks = {
    "GPS":    tele.has_gps(df),
    "IMU":    tele.has_imu(df),
    "Gyro":   tele.has_gyro(df),
    "OBD":    tele.has_obd(df),
    "Magneto": tele.has_magneto(df),
}
for sensor, present in checks.items():
    print(f"{sensor:8s} {'yes' if present else 'no'}")

## Visualise one trip

Pick the trip with the largest row count to have a nice time series.

In [ ]:
import matplotlib.pyplot as plt

trip_id = df['trip_id'].value_counts().index[0]
trip = df[df['trip_id'] == trip_id].copy()
trip['t_s'] = (trip['ts'] - trip['ts'].iloc[0]).dt.total_seconds()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# Trajectory
gps = trip.dropna(subset=['lat', 'lon'])
ax[0].plot(gps['lon'], gps['lat'], linewidth=0.6)
ax[0].set(xlabel='lon', ylabel='lat', title=f'trajectory — trip {trip_id}')
ax[0].set_aspect('equal', 'datalim')

# Speed
ax[1].plot(trip['t_s'], trip['speed_mps'], linewidth=0.5)
ax[1].set(xlabel='t (s)', ylabel='speed (m/s)', title='ground speed (GNSS)')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)

for col, c in zip(['ax_mps2', 'ay_mps2', 'az_mps2'], ['tab:red', 'tab:green', 'tab:blue']):
    ax[0].plot(trip['t_s'], trip[col], linewidth=0.4, color=c, label=col, alpha=0.7)
ax[0].set(ylabel='m/s²', title=f'IMU (trip {trip_id})')
ax[0].legend(loc='upper right', fontsize=8)

for col, c in zip(['gx_rad_s', 'gy_rad_s', 'gz_rad_s'], ['tab:red', 'tab:green', 'tab:blue']):
    ax[1].plot(trip['t_s'], trip[col], linewidth=0.4, color=c, label=col, alpha=0.7)
ax[1].set(xlabel='t (s)', ylabel='rad/s', title='gyroscope')
ax[1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## Validate

AEGIS has GNSS + accelerometer + gyroscope → it matches the `full` profile
(the strictest of the three Telemachus profiles).

In [ ]:
report = tele.validate(df, profile="full")
print(report)

## Where to go next

- **Other open datasets** — STRIDE (Bangladesh, smartphone, 100 Hz) on [Zenodo 19609053](https://doi.org/10.5281/zenodo.19609053), RS3 synthetic (Le Havre) on [Zenodo 19609057](https://doi.org/10.5281/zenodo.19609057).
- **Convert your own data** — `tele convert aegis|pvs|stride <raw-dir> -o <out-dir>`, or write your own adapter: [guide/writing-adapter](https://telemachus3.org/guide/writing-adapter/).
- **Specification** — [SPEC-01 record format](https://github.com/telemachus3/telemachus/blob/main/spec/SPEC-01-record-format.md), [SPEC-02 manifest](https://github.com/telemachus3/telemachus/blob/main/spec/SPEC-02-manifest.md).
- **Cite** — Edet, S. (2026). *Telemachus Specification v0.8*. Zenodo. DOI [10.5281/zenodo.19609019](https://doi.org/10.5281/zenodo.19609019).
- **Feedback / RFC** — comments on the v0.9 draft welcome in [GitHub Discussions](https://github.com/telemachus3/telemachus/discussions).